# 含硫 N 体模型批量工作流

在一个单元中检查全部参数，预览实际路径，然后执行。每个模型保留最近一次成功结果；本次日志与汇总在下次运行时覆盖。

In [1]:
from pathlib import Path
import importlib.util
root = Path.cwd().resolve()
project_root = next((p for p in (root, *root.parents)
                     if (p / "src" / "accrediff").is_dir()), None)
if project_root is None:
    raise RuntimeError("Could not locate the AccreDiff repository root.")
batch_dir = project_root / 'kong_2027_backup' / 'work_flow_S_batch'
spec = importlib.util.spec_from_file_location('accrediff_backup_batch', batch_dir / 'batch_core.py')
batch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(batch)
print(f'Batch notebook: {batch_dir}')


Batch notebook: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch


## 1. 模型与各模块参数

In [2]:
# 所有可调值集中在此单元；模型文件夹须位于 DATA_ROOT 下。
MODEL_LIST = ['TSMO_01']
DATA_ROOT = 'notebooks/TSM_S3'
OUTPUT_ROOT = 'kong_2027_backup/work_flow_S_batch/results'  # 指定全部模型结果的根目录
SCENARIO_ID = 'baseline'
LAST_STAGE = 'E_01'         # 可截断于任一阶段；前面的阶段保持启用
TIMEOUT_SECONDS = {'default': 900, 'D_01': 14400}

# 相同判据仅定义一次；C_02 的局部熔融压力比独立设置。
GAS_TIME = '5.0 Myr'
C_COMMON = {'GLOBAL_MASS_THRESHOLD': 0.1, 'IMPACT_RATIO_RANGE': (0.1, 0.5)}

# 每个阶段列出完整科学/显示参数；路径、MODEL_ID 由执行器逐模型展开。
EXECUTION_PLAN = [
    {'stage': 'A_01', 'notebook': 'A_01_meteorite_bulk_composition_s.ipynb', 'enabled': True,
     'parameters': {'SULFUR_WT': {'EF': 0.0, 'EC': 2.378344, 'OC': 4.429226, 'CI': 11.520290},
                    'EF_ENRICHMENT_FACTOR': 1.5, 'EF_DEPLETION_FACTOR': 0.6}},
    {'stage': 'B_01', 'notebook': 'B_01_embryo_bulk_composition_s.ipynb', 'enabled': True,
     'parameters': {'GAS_TIME': GAS_TIME, 'INITIAL_TIME': '0.0 Myr',
                    'EF_MAX_AU': 0.9747, 'EC_MAX_AU': 1.0674, 'OC_MAX_AU': 3.0,
                    'CI_MIN_AU': 2.0, 'EMBRYO_DISPLAY_THRESHOLD': 0.05}},  # M Earth; plot only
    {'stage': 'B_02', 'notebook': 'B_02_accretion_history_s.ipynb', 'enabled': True,
     'parameters': {'GAS_TIME': GAS_TIME, 'PLANET_MASS_THRESHOLD': 0.01}},  # M Earth
    {'stage': 'C_01', 'notebook': 'C_01_small_impact_s.ipynb', 'enabled': True,
     'parameters': {'P_RATIO': 0.6, **C_COMMON, 'COOLING_DURATION_MYR': 5.0}},
    {'stage': 'C_02', 'notebook': 'C_02_giant_impact_s.ipynb', 'enabled': True,
     'parameters': {**C_COMMON, 'PARTIAL_PRESSURE_RATIO': 0.6}},
    {'stage': 'D_01', 'notebook': 'D_01_differentiation_s.ipynb', 'enabled': True,
     'parameters': {
         'ROOT_SEARCH': 'thorough', 'SMALL_CORE_EQ_FRACTION': 0.7, 'GLOBAL_CORE_EQ_FRACTION': 0.7,
         'GLOBAL_TARGET_MANTLE_FACTOR': 0.05, 'PARTIAL_CORE_EQ_FRACTION': 0.3,
         'OXYGEN_KD_TOL': 1e-8, 'OXYGEN_KD_FALLBACK_TOL': 1.5, # residual in dex
         'MASS_RTOL': 1e-5, 'CONSERVATION_RTOL': 1e-8, 'CONSERVATION_ATOL': 1e-12,
     }},
    {'stage': 'E_01', 'notebook': 'E_01_bulk_comparison_s.ipynb', 'enabled': True,
     'parameters': {
         'SAVE_FIGURES': False,
         'PLOT_DEFAULT_YLIMS': {'Mass': (0.0, 1.15), 'Mg#': (0.68, 1.00), 'CMF': (0.15, 0.38)},
         'FIGURE_SIZE_MM': (183, 148),
         # References from E_00_CMF.ipynb; observational benchmarks, not fit variables.
         'METRIC_REFERENCE': {
             'Mass': {'Earth': 1.0000, 'Mars': 0.1074},
             'Mg#': {'Earth': 0.890, 'Mars': 0.775},
             'CMF': {'Earth': 0.320, 'Mars': 0.200},
         },
         'METRIC_REFERENCE_RANGE': {
             'Mass': {'Earth': None, 'Mars': None},
             'Mg#': {'Earth': (0.801, 0.979), 'Mars': (0.698, 0.853)},
             'CMF': {'Earth': (0.288, 0.352), 'Mars': (0.180, 0.220)},
         },
     }},
]
for item in EXECUTION_PLAN:
    item['enabled'] = item['enabled'] and list(batch.STAGE_BY_NAME).index(item['stage']) <= list(batch.STAGE_BY_NAME).index(LAST_STAGE)

CONFIG = {
    'data_root': DATA_ROOT, 'output_root': OUTPUT_ROOT, 'models': MODEL_LIST,
    'scenario_id': SCENARIO_ID, 'execution_plan': EXECUTION_PLAN,
    'timeouts': TIMEOUT_SECONDS,
}


## 2. 预览展开参数与路径

In [3]:
batch.preview_plan(CONFIG)


Scenario baseline; models ['TSMO_01']
Data: /Users/kong/Research/code_zju/AccreDiff/notebooks/TSM_S3
Output: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results
每次从 A_01 重新计算；同模型成功后覆盖旧结果，失败时保留旧结果。

A_01 A_01_meteorite_bulk_composition_s.ipynb (本批次共用)
  PROJECT_ROOT = '/Users/kong/Research/code_zju/AccreDiff'
  WORKFLOW_DIR = '/Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results/.staging/<attempt_id>/shared/stage_workspaces/A_01'
  SULFUR_WT = {'EF': 0.0, 'EC': 2.378344, 'OC': 4.429226, 'CI': 11.52029}
  EF_ENRICHMENT_FACTOR = 1.5
  EF_DEPLETION_FACTOR = 0.6
  OUTPUT_FILE = '/Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results/.staging/<attempt_id>/shared/stage_workspaces/A_01/Tables/Meteorites_composition.csv'
  OUTPUT_INVENTORY_FILE = '/Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results/.staging/<attempt_id>/shared/stage_workspaces/A_01/Tables/Meteorites_inventory_

## 3. 正式运行

执行本单元即启动批次。成功模型覆盖自己的旧结果；失败模型保留旧结果。状态与日志写入 OUTPUT_ROOT/latest_run/，下次运行覆盖。

In [4]:
latest_dir = batch.execute(CONFIG)
print(f"Latest attempt: {latest_dir}")


A_01: passed
TSMO_01 / B_01: passed
TSMO_01 / B_02: passed
TSMO_01 / C_01: passed
TSMO_01 / C_02: passed
TSMO_01 / D_01: passed
TSMO_01 / E_01: passed
Batch status: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results/latest_run/batch_status.csv
Latest attempt: /Users/kong/Research/code_zju/AccreDiff/kong_2027_backup/work_flow_S_batch/results/latest_run
